# 02 · Attempt Propensity π(X)

**Purpose:** Estimate P(FG Attempt | game state X) via rolling-origin multinomial
logistic regression over actions (FG, Punt, Go-for-it). Produces inverse-probability
weights (IPW) used by notebook 03 to correct for selection bias in the outcome model.

**Inputs:**
- `data/fg_all.csv` (from notebook 01)

**Outputs:**
- `reports/attempt_pi/attempt_pi_oof_predictions_final.csv`
- `reports/attempt_pi/attempt_pi_metrics_by_season.csv`

**IPW Weight Pipeline (5 steps):**
1. Clip propensities to [0.02, 0.98]
2. Stabilized IPW: w_raw = prevalence / p_clipped
3. Hájek season-normalize (mean weight = 1 per season)
4. 3σ cap (remove extreme outliers)
5. Re-normalize (mean = 1 again)

In [1]:
# ============================================================
# 1. Parameters
# ============================================================
PROJECT_ROOT <- sub('[/\\\\][^/\\\\]*$', '', getwd())

data_dir    <- file.path(PROJECT_ROOT, 'data')
reports_dir <- file.path(PROJECT_ROOT, 'reports')
reports_pi_dir <- file.path(reports_dir, 'attempt_pi')

# Rolling-origin window size (train on prior N seasons)
WINDOW        <- 3L
# First season to generate OOF predictions for
OOF_START     <- 2015L
# Probability clipping bounds
CLIP_MIN      <- 0.02
CLIP_MAX      <- 0.98
# Distance spline basis. These MUST match notebook 03 (DIST_KNOTS / DIST_BOUNDS
# there) so that the propensity model and the outcome model resolve distance on
# the same basis; a mismatch would make the IPW weights a function of a distance
# representation the outcome model never sees.
DIST_KNOTS    <- c(28, 38, 48, 58)
DIST_BOUNDS   <- c(18, 70)

set.seed(20240517)
if (!dir.exists(reports_pi_dir)) dir.create(reports_pi_dir, recursive = TRUE)
message('PROJECT_ROOT: ', PROJECT_ROOT)

PROJECT_ROOT: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model



In [2]:
# ============================================================
# 2. Imports
# ============================================================
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'splines', 'splines2', 'pROC', 'nnet'
)
installed <- rownames(installed.packages())
for (pkg in dependencies) {
  if (!pkg %in% installed) install.packages(pkg)
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}
message('Libraries loaded.')

Libraries loaded.



In [3]:
# ============================================================
# 3. Helper Functions
# ============================================================

# B-spline basis on kick distance
build_distance_bs <- function(x, knots = DIST_KNOTS, bknots = DIST_BOUNDS, degree = 3) {
  bs <- splines2::bSpline(x, knots = knots, Boundary.knots = bknots,
                           degree = degree, intercept = FALSE)
  bs <- as.data.frame(bs)
  names(bs) <- paste0('bs_', seq_len(ncol(bs)))
  bs
}

# Effective Sample Size
ess <- function(w) {
  w <- w[is.finite(w)]
  if (!length(w)) return(NA_real_)
  (sum(w)^2) / sum(w^2)
}

brier   <- function(y, p) mean((y - p)^2, na.rm = TRUE)
logloss <- function(y, p, eps = 1e-15)
  -mean(y * log(pmin(pmax(p, eps), 1-eps)) + (1-y) * log(1 - pmin(pmax(p, eps), 1-eps)),
        na.rm = TRUE)

## 4. Load Data & Build Decision Frame

In [4]:
fg_path <- file.path(data_dir, 'fg_all.csv')
stopifnot(file.exists(fg_path))
fg_all <- readr::read_csv(fg_path, show_col_types = FALSE)
message('fg_all loaded: ', nrow(fg_all), ' rows | seasons: ',
        min(fg_all$season), '-', max(fg_all$season))

fg_all loaded: 137668 rows | seasons: 2000-2025



In [5]:
# Build 4th-down decision frame (exclude PATs)
# action levels: go_for_it (reference), punt, FG
decision <- fg_all %>%
  filter(
    is_pat == 0L,
    play_type_original %in% c('field_goal', 'pass', 'run', 'punt'),
    !is.na(kick_distance),
    kick_distance <= 70,
    !is.na(game_seconds_remaining)
  ) %>%
  mutate(
    attempt_fg = as.integer(play_type_original == 'field_goal'),
    action = dplyr::case_when(
      play_type_original == 'field_goal' ~ 'FG',
      play_type_original == 'punt'       ~ 'punt',
      TRUE                               ~ 'go_for_it'
    ),
    # Score state flags
    go_ahead       = as.integer(!is.na(score_differential) & score_differential >= -2 & score_differential <= 0),
    one_score_down = as.integer(!is.na(score_differential) & score_differential <= -4 & score_differential >= -8),
    to_tie         = as.integer(score_differential == -3),
    extend_lead    = as.integer(!is.na(score_differential) & score_differential >= 1),
    two_score_down = as.integer(!is.na(score_differential) & score_differential <= -9 & score_differential >= -11),
    # Venue/weather (coerce NAs to 0 for sparse flags)
    indoors      = dplyr::coalesce(as.integer(indoors),      0L),
    is_turf      = dplyr::coalesce(as.integer(is_turf),      0L),
    high_altitude = dplyr::coalesce(as.integer(high_altitude), 0L)
  )

message('Decision frame: ', nrow(decision), ' plays | ',
        sum(decision$attempt_fg), ' FG, ',
        sum(decision$action == 'punt'), ' punt, ',
        sum(decision$action == 'go_for_it'), ' go')

Decision frame: 53759 plays | 26764 FG, 15253 punt, 11742 go



In [6]:
# ============================================================
# 5. Feature Construction
# ============================================================

# B-spline basis on kick distance
bs_X <- build_distance_bs(decision$kick_distance)

# Natural spline on game seconds remaining
ns_time <- splines::ns(decision$game_seconds_remaining, df = 4)
colnames(ns_time) <- paste0('nst_', seq_len(ncol(ns_time)))

# Quarter-time interaction features
quarter_sec  <- as.numeric(decision$quarter_seconds_remaining)
log_qtr_time <- log1p(pmax(quarter_sec, 0))
q2_flag      <- as.integer(as.integer(decision$qtr) == 2)
q4_flag      <- as.integer(as.integer(decision$qtr) == 4)

X <- cbind(
  decision,
  as.data.frame(bs_X),
  as.data.frame(ns_time),
  log_qtr_time = log_qtr_time,
  q2_flag      = q2_flag,
  q4_flag      = q4_flag,
  q2_log_time  = q2_flag * log_qtr_time,
  q4_log_time  = q4_flag * log_qtr_time
)

X$season_num <- as.integer(X$season)
X$action <- factor(X$action, levels = c('go_for_it', 'punt', 'FG'))

# Centered linear season trend (reviewer Main-3: coaching decisions have shifted
# over time, so the propensity model must carry a season term).
# nnet::multinom is fixed-effects only, and each rolling fold trains on just 3
# seasons, so a spline or random effect on season is not identifiable per fold.
# A linear trend centered on the training window costs 1 parameter per class and
# captures within-window drift. The value is filled per fold inside the rolling
# loop below (centering uses the TRAINING window mean, applied to the test
# season as well); initialised here so the term survives the names(X) filter.
X$season_c <- 0

# Model formula
bs_terms  <- paste0('bs_',  seq_len(ncol(bs_X)))
ns_terms  <- paste0('nst_', 4)
base_terms <- c(
  bs_terms, paste0('nst_', seq_len(4)),
  'wind_z', 'temp_z', 'ydstogo', 'leverage_z', 'season_c',
  'indoors', 'is_turf', 'high_altitude',
  'is_ot',
  'go_ahead', 'one_score_down', 'to_tie', 'extend_lead', 'two_score_down',
  'q2_log_time', 'q4_log_time'
)
# Drop columns absent from X
base_terms <- base_terms[base_terms %in% names(X)]
form_multinom <- as.formula(paste('action ~', paste(base_terms, collapse = ' + ')))

message('Feature matrix built: ', nrow(X), ' rows, ', ncol(X), ' cols')
message('Formula terms: ', length(base_terms))

Feature matrix built: 53759 rows, 107 cols



Formula terms: 27



## 6. Rolling-Origin OOF Multinomial Fit

For each season s ≥ 2015, train on the prior 3 seasons, predict on season s.
This prevents data leakage and captures era drift in coaching behavior.

In [7]:
SEASONS_ALL <- sort(unique(X$season_num))
SEASONS_OOF <- SEASONS_ALL[SEASONS_ALL >= OOF_START]

all_preds    <- list()
metrics_rows <- list()

for (s in SEASONS_OOF) {
  train_seasons <- seq(max(min(SEASONS_ALL), s - WINDOW), s - 1)
  train_idx <- which(X$season_num %in% train_seasons)
  test_idx  <- which(X$season_num == s)
  if (length(train_idx) < 100 || length(test_idx) == 0) next

  train <- X[train_idx, , drop = FALSE]
  test  <- X[test_idx,  , drop = FALSE]

  # Center the season trend on the TRAINING window mean, and apply the same
  # centering constant to the test season. The test fold therefore sits one
  # step beyond the training centre (season_c = +2 for a 3-season window),
  # which is exactly the one-season-ahead drift extrapolation we want.
  season_center   <- mean(train$season_num, na.rm = TRUE)
  train$season_c  <- train$season_num - season_center
  test$season_c   <- test$season_num  - season_center

  m_multi <- tryCatch(
    nnet::multinom(form_multinom, data = train, trace = FALSE),
    error = function(e) { message('Season ', s, ' failed: ', e$message); NULL }
  )

  p_multi <- rep(NA_real_, nrow(test))
  act_pred <- rep(NA_character_, nrow(test))

  if (!is.null(m_multi)) {
    preds_mat <- tryCatch(predict(m_multi, newdata = test, type = 'probs'), error = function(e) NULL)
    if (!is.null(preds_mat) && 'FG' %in% colnames(preds_mat)) {
      p_multi  <- as.numeric(preds_mat[, 'FG'])
      act_pred <- apply(preds_mat, 1, function(r) colnames(preds_mat)[which.max(r)])
    }
  }

  y_bin       <- test$attempt_fg
  target_prev <- mean(train$attempt_fg, na.rm = TRUE)

  # Clip (numerical stability only — NOT weight trimming)
  eps <- .Machine$double.eps^0.5
  p_clip <- pmin(pmax(p_multi, eps), 1 - eps)

  # Stabilized raw IPW
  w_raw <- ifelse(
    y_bin == 1L,
    target_prev / p_clip,
    (1 - target_prev) / (1 - p_clip)
  )

  # Hájek normalize within fold (mean = 1)
  w_hajek <- w_raw / mean(w_raw, na.rm = TRUE)

  fold_preds <- tibble::tibble(
    season               = s,
    season_c             = test$season_c,
    game_id              = test$game_id,
    play_id              = test$play_id,
    attempt_fg           = y_bin,
    action_actual        = as.character(test$action),
    action_pred          = as.character(act_pred),
    p_hat_multinom       = p_multi,
    p_hat_attempt_clipped = p_clip,
    p_target_multinom    = target_prev,
    weight_multinom_raw  = w_raw,
    weight_multinom_hajek = w_hajek
  )
  all_preds[[as.character(s)]] <- fold_preds

  auc_val <- tryCatch(as.numeric(pROC::auc(y_bin, p_multi, quiet = TRUE)), error = function(e) NA_real_)
  metrics_rows[[paste0('s', s)]] <- tibble::tibble(
    season = s, model = 'multinom',
    auc = auc_val, brier = brier(y_bin, p_multi), logloss = logloss(y_bin, p_multi),
    n_train = length(train_idx), n_test = length(test_idx),
    target_prev = target_prev,
    ess_hajek = ess(w_hajek[y_bin == 1L])
  )

  message(sprintf('Season %d: AUC=%.3f | Brier=%.3f | train_n=%d | test_n=%d',
    s, auc_val, brier(y_bin, p_multi), length(train_idx), length(test_idx)))
}

preds_oof   <- dplyr::bind_rows(all_preds) %>%
  mutate(game_id = as.character(game_id), play_id = as.character(play_id))
metrics_oof <- dplyr::bind_rows(metrics_rows)

message('\nOOF predictions: ', nrow(preds_oof), ' rows')
print(metrics_oof)

Season 2015: AUC=0.961 | Brier=0.075 | train_n=5975 | test_n=1991



Season 2016: AUC=0.958 | Brier=0.078 | train_n=5948 | test_n=1955



Season 2017: AUC=0.957 | Brier=0.079 | train_n=5886 | test_n=2008



Season 2018: AUC=0.959 | Brier=0.080 | train_n=5954 | test_n=1959



Season 2019: AUC=0.950 | Brier=0.089 | train_n=5922 | test_n=1982



...

Season 2023: AUC=0.944 | Brier=0.095 | train_n=6295 | test_n=2187



Season 2024: AUC=0.940 | Brier=0.098 | train_n=6508 | test_n=2228



Season 2025: AUC=0.943 | Brier=0.098 | train_n=6561 | test_n=2274




OOF predictions: 22879 rows



# A tibble: 11 × 9
   season model      auc  brier logloss n_train n_test target_prev ess_hajek
    <int> <chr>    <dbl>  <dbl>   <dbl>   <int>  <int>       <dbl>     <dbl>
 1   2015 multinom 0.961 0.0755   0.247    5975   1991       0.522     67.2 
 2   2016 multinom 0.958 0.0782   0.255    5948   1955       0.521    211.  
 3   2017 multinom 0.957 0.0792   0.257    5886   2008       0.528     92.7 
 4   2018 multinom 0.959 0.0797   0.256    5954   1959       0.529    132.  
 5   2019 multinom 0.950 0.0890   0.282    5922   1982       0.524    127.  
 6   2020 multinom 0.944 0.0946   0.301    5949   1974       0.516      1.22
 7   2021 multinom 0.940 0.0984   0.310    5915   2175       0.511    175.  
 8   2022 multinom 0.938 0.100    0.314    6131   2146       0.507    208.  
 9   2023 multinom 0.944 0.0947   0.299    6295   2187       0.508    333.  
10   2024 multinom 0.940 0.0981   0.311    6508   2228       0.505     24.3 
11   2025 multinom 0.943 0.0982   0.308    6561   2274   

## 7. Full IPW Weight Pipeline

Applied to all rows in the OOF predictions dataframe:
1. Clip to [0.02, 0.98]
2. Stabilized IPW
3. Hájek season-normalize
4. 3σ cap
5. Re-normalize (mean = 1)

In [8]:
# Step 1: Clip propensities
preds_oof <- preds_oof %>%
  mutate(
    p_hat_attempt_clipped = pmin(pmax(p_hat_multinom, CLIP_MIN), CLIP_MAX),
    # Step 2: Stabilized IPW
    w_ipw_raw = ifelse(
      attempt_fg == 1L,
      p_target_multinom / p_hat_attempt_clipped,
      (1 - p_target_multinom) / (1 - p_hat_attempt_clipped)
    )
  )

# Step 3: Hájek normalize within season
preds_oof <- preds_oof %>%
  group_by(season) %>%
  mutate(w_clip_hajek = w_ipw_raw / mean(w_ipw_raw, na.rm = TRUE)) %>%
  ungroup()

# Steps 4 & 5: 3σ cap then re-normalize (legacy canonical weight)
preds_oof <- preds_oof %>%
  group_by(season) %>%
  mutate(
    w_mean = mean(w_clip_hajek, na.rm = TRUE),
    w_sd   = sd(w_clip_hajek,   na.rm = TRUE),
    w_cap  = w_mean + 3 * w_sd,
    w_capped = pmin(w_clip_hajek, w_cap),
    w_ipw_final = w_capped / mean(w_capped, na.rm = TRUE)
  ) %>%
  ungroup() %>%
  select(-w_mean, -w_sd, -w_cap, -w_capped)

In [9]:
# ============================================================
# 8. Weight Diagnostics
# ============================================================
diag <- preds_oof %>%
  filter(attempt_fg == 1L) %>%
  summarise(
    n         = n(),
    min_w     = min(w_ipw_final, na.rm = TRUE),
    median_w  = median(w_ipw_final, na.rm = TRUE),
    mean_w    = mean(w_ipw_final, na.rm = TRUE),
    p99_w     = quantile(w_ipw_final, 0.99, na.rm = TRUE),
    max_w     = max(w_ipw_final, na.rm = TRUE),
    sd_w      = sd(w_ipw_final, na.rm = TRUE),
    ess       = ess(w_ipw_final),
    ess_ratio = ess / n
  )
cat('\n=== FG Attempt IPW Weight Summary ===\n')
print(diag)

diag_by_season <- preds_oof %>%
  filter(attempt_fg == 1L) %>%
  group_by(season) %>%
  summarise(
    n = n(), mean_w = mean(w_ipw_final, na.rm = TRUE),
    max_w = max(w_ipw_final, na.rm = TRUE), ess = ess(w_ipw_final),
    .groups = 'drop'
  )
print(diag_by_season)


=== FG Attempt IPW Weight Summary ===


# A tibble: 1 × 9
      n min_w median_w mean_w p99_w max_w  sd_w   ess ess_ratio
  <int> <dbl>    <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>     <dbl>
1 11764 0.630    0.708  0.963  6.55  8.08 0.875 6444.     0.548


# A tibble: 11 × 5
   season     n mean_w max_w   ess
    <int> <int>  <dbl> <dbl> <dbl>
 1   2015  1033  0.955  7.80  590.
 2   2016  1050  0.912  7.24  634.
 3   2017  1065  0.946  8.08  577.
 4   2018   989  0.954  6.95  621.
 5   2019  1018  0.936  7.45  535.
 6   2020  1015  1.03   6.68  559.
 7   2021  1076  0.910  7.29  624.
 8   2022  1105  0.988  7.26  588.
 9   2023  1107  0.982  6.42  641.
10   2024  1166  0.963  8.05  585.
11   2025  1140  1.01   7.08  552.


In [10]:
# ============================================================
# 9. Save Outputs
# ============================================================
out_preds   <- file.path(reports_pi_dir, 'attempt_pi_oof_predictions_final.csv')
out_metrics <- file.path(reports_pi_dir, 'attempt_pi_metrics_by_season.csv')

readr::write_csv(preds_oof,   out_preds)
readr::write_csv(metrics_oof, out_metrics)

message('\n=== Outputs Written ===')
message('OOF predictions: ', out_preds, ' (', nrow(preds_oof), ' rows)')
message('Season metrics:  ', out_metrics)


=== Outputs Written ===



OOF predictions: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/attempt_pi/attempt_pi_oof_predictions_final.csv (22879 rows)



Season metrics:  X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/attempt_pi/attempt_pi_metrics_by_season.csv

